# 01 — Bronze Daily Ingest: One TED Package per Day

**Purpose**

Daily-run notebook. Downloads ONE day's TED package, extracts the XML to Bronze, deletes the `.tar.gz` to save space. Defaults to today; override the widget to fetch a specific date.

**Idempotent**

If the day's folder already exists in Bronze with XML files in it, the notebook skips the download. Safe to re-run.

**Calendar source**

The TED release-calendar URL is behind AWS WAF and returns 202 + empty body to non-browser clients. We work around it by reading a manually-downloaded CSV from the Volume at `/Volumes/workspace/default/lakehouse/bronze/calendar/ted_calendar_YYYY.csv`. Download once per year from `https://ted.europa.eu/en/release-calendar/-/download/file/CSV/YYYY` in a regular browser, upload to that Volume path.

## Cell 1 — Imports & date widget

In [0]:
# ── CELL 1 — Imports & date widget ───────────────────────────────────────────
import os
import csv
import urllib.request
import tarfile
from datetime import datetime, date
from io import StringIO

dbutils.widgets.text("run_date", "", "Run date YYYY-MM-DD (blank = today)")
_widget = dbutils.widgets.get("run_date").strip()
run_date = datetime.strptime(_widget, "%Y-%m-%d").date() if _widget else date.today()

VOLUME_ROOT   = "/Volumes/workspace/default/lakehouse/bronze/ted"
CALENDAR_ROOT = "/Volumes/workspace/default/lakehouse/bronze/calendar"

print(f"Run date     : {run_date}")
print(f"Bronze root  : {VOLUME_ROOT}")
print(f"Calendar root: {CALENDAR_ROOT}")


## Cell 2 — Publication-number lookup (from local CSV)

The CSV maps a calendar date to TED's OJS publication number (e.g. 2026-06-12 → `202600112`). Format as TED serves it: `OJS,Publication date` with dates in `DD/MM/YYYY`.

In [0]:
# ── CELL 2 — Publication-number lookup (from local CSV) ──────────────────────
_calendar_cache = {}  # year -> {date: ojs_string}

def _parse_csv_date(s):
    """Try common formats; return a date or None."""
    s = s.strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    return None

def _load_year_calendar(year):
    """Read the local TED calendar CSV for `year` and return {date: ojs_string}."""
    if year in _calendar_cache:
        return _calendar_cache[year]

    path = f"{CALENDAR_ROOT}/ted_calendar_{year}.csv"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {path}. Download once per year from "
            f"https://ted.europa.eu/en/release-calendar/-/download/file/CSV/{year} "
            f"(in a regular browser — the URL is WAF-blocked for scripts) and upload to that Volume path."
        )

    mapping = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        headers = [h.strip().lower() for h in next(reader, [])]
        # Default column order for TED's CSV is OJS first, date second.
        date_idx, ojs_idx = 1, 0
        for i, h in enumerate(headers):
            if "date" in h:                                    date_idx = i
            if "ojs" in h or "issue" in h or "number" in h:    ojs_idx  = i
        for row in reader:
            if len(row) <= max(date_idx, ojs_idx):
                continue
            d = _parse_csv_date(row[date_idx])
            digits = "".join(c for c in row[ojs_idx] if c.isdigit())
            if d and digits:
                mapping[d] = f"{year}{int(digits):05d}"

    _calendar_cache[year] = mapping
    print(f"Loaded {year} calendar from {path}: {len(mapping)} publication days")
    return mapping

def get_publication_number(target_date):
    """Return TED publication ID (e.g. '202600112'), or None if not a publishing day."""
    return _load_year_calendar(target_date.strftime("%Y")).get(target_date)


## Cell 3 — Ingest one day

Steps:

1. Skip if the day's folder already exists in Bronze with XML files in it.
2. Look up the publication number.
3. Download the archive from `https://ted.europa.eu/packages/daily/{pub}`.
4. Extract the XML.
5. Delete the archive to save space (~18 MB per day; the XML is what Silver needs).

In [0]:
# ── CELL 3 — Ingest the run_date ─────────────────────────────────────────────
def folder_has_xml(folder):
    """True if the folder exists and contains at least one .xml file."""
    if not os.path.exists(folder):
        return False
    for _, _, files in os.walk(folder):
        for f in files:
            if f.endswith(".xml"):
                return True
    return False

YYYY     = run_date.strftime("%Y")
MM       = run_date.strftime("%m")
YYYYMMDD = run_date.strftime("%Y%m%d")
folder   = f"{VOLUME_ROOT}/{YYYY}/{MM}/{YYYYMMDD}"

if folder_has_xml(folder):
    print(f"SKIP {run_date}: already in Bronze at {folder}")
else:
    pub = get_publication_number(run_date)
    if not pub:
        print(f"SKIP {run_date}: not a publishing day (no entry in TED calendar)")
    else:
        dbutils.fs.mkdirs(folder)
        archive = f"{folder}/{pub}.tar.gz"

        print(f"Downloading {run_date} (publication {pub})...")
        urllib.request.urlretrieve(f"https://ted.europa.eu/packages/daily/{pub}", archive)

        print("Extracting...")
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall(folder)

        os.remove(archive)

        # Count what we just wrote
        xml_count = sum(
            1
            for _, _, files in os.walk(folder)
            for f in files
            if f.endswith(".xml")
        )
        print(f"OK {run_date}: wrote {xml_count:,} XML files to {folder}")


## Cell 4 — Verify what Bronze now contains for this date

In [0]:
# ── CELL 4 — Verify Bronze contents for run_date ─────────────────────────────
if os.path.exists(folder):
    xml_count = sum(
        1
        for _, _, files in os.walk(folder)
        for f in files
        if f.endswith(".xml")
    )
    print(f"{run_date}: {xml_count:,} XML files in {folder}")
else:
    print(f"{run_date}: no folder at {folder}")


## Next step

Run **`02_parse_notices`** with the same `run_date` to parse this day's XML into Silver.